# AML Machine Learning: Modellvergleich zur Betrugserkennung

**Lernziel:** Sie trainieren und evaluieren verschiedene ML‑Modelle auf einem synthetischen AML‑Datensatz:
- Logistische Regression (Baseline)
- Random Forest
- XGBoost
- Vergleich der Leistungsmetriken (Precision, Recall, ROC‑AUC)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_score, recall_score, roc_auc_score, classification_report

# Synthetische Daten erzeugen
np.random.seed(42)
n = 5000
amount = np.random.gamma(2, 500, n)
hour = np.random.randint(0, 24, n)
weekend = np.random.choice([0, 1], n, p=[0.7, 0.3])
tx_count_7d = np.random.poisson(5, n)

# Zielvariable: Betrug (abhängig von amount, hour, weekend)
fraud_prob = 1 / (1 + np.exp(-(0.001 * amount + 0.2 * (hour > 22) + 0.4 * weekend - 2)))
fraud = np.random.binomial(1, fraud_prob)

df = pd.DataFrame({'amount': amount, 'hour': hour, 'weekend': weekend, 'tx_count_7d': tx_count_7d, 'fraud': fraud})
print(f"Betrugsrate: {df['fraud'].mean():.2%}")
print(df.head())

In [ ]:
# Train/Test Split
X = df.drop('fraud', axis=1)
y = df['fraud']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardisierung
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'),
    'XGBoost': XGBClassifier(scale_pos_weight= (len(y)-sum(y))/sum(y), random_state=42, eval_metric='logloss')
}

results = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    results.append({
        'Model': name,
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'ROC‑AUC': roc_auc_score(y_test, y_proba)
    })

results_df = pd.DataFrame(results)
print(results_df.round(4))

## Diskussion

- **XGBoost** liefert oft die beste AUC, ist aber rechenintensiver.
- **Random Forest** ist robuster gegenüber Überanpassung.
- **Logistische Regression** dient als einfache Baseline.

In der Praxis würde man das beste Modell auswählen und weiter optimieren (Hyperparameter‑Tuning).